# Atividade 2 - Integridade e Transformação de Dados

**Conjunto de dados:** *Diabetes 130-US Hospitals for Years 1999–2008* 

**URL:** https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

**Descrição:** O conjunto de dados representa dez anos (1999-2008) de atendimento clínico em 130 hospitais e redes integradas de saúde dos Estados Unidos. Cada linha diz respeito aos registros hospitalares de pacientes diagnosticados com diabetes, que foram submetidos a exames laboratoriais, receberam medicamentos e permaneceram internados por até 14 dias. O objetivo é determinar a readmissão precoce do paciente no prazo de 30 dias após a alta médica. A descrição completa das colunas da tabela está no arquivo `dados_tabela.md`.

**Número de atributos:** O UCI indica 47 atributos, mas na descrição do dataset no UCI está escrito "It includes over 50 features..."

## Leitura dos dados e testes


In [1]:
import pandas as pd

import testes_integridade as ti


data_path = "../../dados/diabetic_data.csv"

# Leitura dos dadoss sem tratar os valores ausentes, para que possamos analisar a ocorrência de cada tipo de ausência.
df = pd.read_csv(data_path, keep_default_na=False)
print(df.shape)
df.head(3)

(10000, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   encounter_id              10000 non-null  int64
 1   patient_nbr               10000 non-null  int64
 2   race                      10000 non-null  str  
 3   gender                    10000 non-null  str  
 4   age                       10000 non-null  str  
 5   weight                    10000 non-null  str  
 6   admission_type_id         10000 non-null  int64
 7   discharge_disposition_id  10000 non-null  int64
 8   admission_source_id       10000 non-null  int64
 9   time_in_hospital          10000 non-null  int64
 10  payer_code                10000 non-null  str  
 11  medical_specialty         10000 non-null  str  
 12  num_lab_procedures        10000 non-null  int64
 13  num_procedures            10000 non-null  int64
 14  num_medications           10000 non-null  int64
 1

In [3]:
ti.check_diabetes_dataset(df)


Problemas encontrados em check_sentinel_values:
Coluna 'race' possui 184 valores '?' (1.8% das linhas)
Coluna 'weight' possui 9725 valores '?' (97.2% das linhas)
Coluna 'payer_code' possui 10000 valores '?' (100.0% das linhas)
Coluna 'medical_specialty' possui 3629 valores '?' (36.3% das linhas)
Coluna 'diag_1' possui 6 valores '?' (0.1% das linhas)
Coluna 'diag_2' possui 63 valores '?' (0.6% das linhas)
Coluna 'diag_3' possui 300 valores '?' (3.0% das linhas)
Coluna 'max_glu_serum' possui 9115 valores 'None' (91.1% das linhas)
Coluna 'A1Cresult' possui 8114 valores 'None' (81.1% das linhas)

Problemas encontrados em check_zero_variance:
Colunas com um único valor: payer_code, acetohexamide, miglitol, examide, citoglipton, glipizide-metformin, glimepiride-pioglitazone, metformin-rosiglitazone, metformin-pioglitazone

Problemas encontrados em check_medication_consistency:
1709 linhas com change='Ch' mas nenhum fármaco com dose Up/Down

Problemas encontrados em check_icd9_format:
Coluna

### Leitura dos resultados

- **`check_sentinel_values`** 
  - O caractere `?` é utilizado de forma sistemática para indicar a ausência de dados. 
  - O termo `None` é usado para indicar que os testes `max_glu_serum` e `A1Cresult` não foram realizados. A não realização do teste é diferentes de o resultado estar ausente. 
  - `payer_code` (100%) e `weight` (97,2%) estão vazias demais para qualquer imputação.

- **`check_zero_variance`** 
  - nove colunas constantes. Oito são medicamentos que nunca foram prescritos.

- **`check_medication_consistency`**
  - 1.709 linhas indicam que houve alteração na medicação, mas nenhuma das 23 colunas de fármacos registra `Up`/`Down`.

- **`check_icd9_format`**
  - Há diversos códigos ICD9 que não seguem o padrão esperado. Isso dificulta a análise de comorbidades, pois o código é a única forma de identificar a doença.

## Análise de valores ausentes

In [4]:
# Considera "?" como valor ausente, mas não considera "None" como ausente
df = pd.read_csv(data_path, na_values=["?"], keep_default_na=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   encounter_id              10000 non-null  int64  
 1   patient_nbr               10000 non-null  int64  
 2   race                      9816 non-null   str    
 3   gender                    10000 non-null  str    
 4   age                       10000 non-null  str    
 5   weight                    275 non-null    str    
 6   admission_type_id         10000 non-null  int64  
 7   discharge_disposition_id  10000 non-null  int64  
 8   admission_source_id       10000 non-null  int64  
 9   time_in_hospital          10000 non-null  int64  
 10  payer_code                0 non-null      float64
 11  medical_specialty         6371 non-null   str    
 12  num_lab_procedures        10000 non-null  int64  
 13  num_procedures            10000 non-null  int64  
 14  num_medications   

#### `diag_2` e `diag_3`: ausência justificada

In [5]:
print("Taxa de ausência de diag_3 por número de diagnósticos:")
for n in range(1, 4):
    df_diag = df[df["number_diagnoses"] == n]
    print(n, (df_diag["diag_3"].isna()).mean())

Taxa de ausência de diag_3 por número de diagnósticos:
1 1.0
2 1.0
3 0.0020161290322580645


Não imputar valores em `diag_2` e `diag_3`.


#### `medical_specialty`: depende de outras colunas

In [6]:
missing = df["medical_specialty"].isna()
print("Taxa de ausência de medical_specialty por origem da admissão:")
print(missing.groupby(df["admission_source_id"]).mean().round(2))

Taxa de ausência de medical_specialty por origem da admissão:
admission_source_id
1     0.24
2     0.94
3     0.74
4     0.68
5     0.59
6     0.79
7     0.33
8     0.00
17    0.24
20    1.00
Name: medical_specialty, dtype: float64


Por ser categórica, uma boa opção é criar uma categoria própria `Desconhecida` para os valores ausentes. 

#### Códigos administrativos

As colunas `admission_type_id`, `discharge_disposition_id` e `admission_source_id` possuem ids associados com descrições mostradas na tabela `diabetic_id.csv`. Mas diversos ids não possuem descrição:

- admission_type_id: 5: Not Available, 6: NULL, 8: Not Mapped
- discharge_disposition_id: 18: NULL, 25: Not Mapped, 26: Unknown/Invalid
- admission_source_id: 9: Not Available, 15: Not Available, 17: NULL, 20: Not Mapped, 21: Unknown/Invalid



**O que fazer:** Recodificar os ids para um único id `Desconhecido`.

#### Imputação de valores numéricos

Nenhuma coluna numérica possui valores ausentes. Portanto, `KNNImputer` e `IterativeImputer` não podem ser aplicados.

## Escala e assimetria dos atributos numéricos

Os atributos `encounter_id`, `patient_nbr`, `admission_type_id`, `discharge_disposition_id` e `admission_source_id` são números, mas representam ids e categorias. Portanto, não devem ser escalados.

In [7]:
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
]

df_num = df[numeric_cols]

df_stats = pd.DataFrame({
    "media": df_num.mean(),
    "desvio": df_num.std(),
    "skew": df_num.skew(),
    "min": df_num.min(),
    "max": df_num.max(),
    "amplitude": df_num.max() - df_num.min(),
})
print(df_stats.round(2))

                    media  desvio   skew  min  max  amplitude
time_in_hospital     4.71    3.15   1.01    1   14         13
num_lab_procedures  46.63   18.25  -0.08    1  114        113
num_procedures       1.48    1.77   1.15    0    6          6
num_medications     14.92    7.93   1.33    1   62         61
number_outpatient    0.07    0.46  12.88    0   12         12
number_emergency     0.04    0.43  31.86    0   22         22
number_inpatient     0.47    1.02   3.76    0   15         15
number_diagnoses     6.76    2.06  -0.60    1    9          8


`number_emergency` (31,9), `number_outpatient` (12,9) e `number_inpatient` (3,8)
são casos extremos de skew. O motivo é que são variáveis de contagem infladas em zero:

In [8]:
print((df_num == 0).mean().round(2).sort_values(ascending=False))

number_emergency      0.97
number_outpatient     0.96
number_inpatient      0.72
num_procedures        0.42
time_in_hospital      0.00
num_lab_procedures    0.00
num_medications       0.00
number_diagnoses      0.00
dtype: float64


### Aplicando o `PowerTransformer`

In [9]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer().set_output(transform="pandas").fit(df_num)
df_t = pt.transform(df_num)

df_stats_new = pd.DataFrame({
    "media": df_t.mean(),
    "desvio": df_t.std(),
    "skew": df_t.skew()
})
print(df_stats_new.round(2))

                    media  desvio  skew
time_in_hospital      0.0     1.0  0.00
num_lab_procedures   -0.0     1.0 -0.08
num_procedures       -0.0     1.0  0.15
num_medications       0.0     1.0  0.01
number_outpatient     0.0     1.0  4.74
number_emergency     -0.0     1.0  5.76
number_inpatient      0.0     1.0  1.01
number_diagnoses      0.0     1.0 -0.26


In [10]:
(df_stats["skew"].abs() - df_stats_new["skew"].abs()).round(2)


time_in_hospital       1.00
num_lab_procedures    -0.00
num_procedures         0.99
num_medications        1.31
number_outpatient      8.14
number_emergency      26.11
number_inpatient       2.75
number_diagnoses       0.34
Name: skew, dtype: float64

### O que fazer com as colunas infladas em zero

- **Binarizar:** `had_emergency = number_emergency > 0`. Indica se houve alguma emergência, o que ocorre apenas 2,8% dos casos.
- **Discretizar em poucas faixas:** `0`, `1`, `2+` preserva um pouco mais de informação.
- **Usar um modelo baseado em árvore**: separa `0` de `> 0` naturalmente.
